# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [1]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 80.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=740d6046-306d-4fbc-afb0-1883f80d9f21
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 73.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 206MB/s]


In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [3]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [4]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [5]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    x = x.detach()
    delta = torch.zeros_like(x)
    if rand_init:
        delta.uniform_(-eps, eps)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = (delta + alpha * grad.sign()).clamp(-eps, eps)
        delta = clip_box(x, delta)
    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    x = x.detach()
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.randn_like(x)
        delta = delta / l2_norm(delta).clamp_min(1e-12)
        radius = torch.rand(x.shape[0], 1, 1, 1, device=x.device, dtype=x.dtype)
        radius = radius.pow(1.0 / x[0].numel())
        delta = delta * eps * radius
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad / l2_norm(grad).clamp_min(1e-12)
        scale = (eps / l2_norm(delta).clamp_min(1e-12)).clamp(max=1.0)
        delta = clip_box(x, delta * scale)
    return (x + delta).detach()

In [6]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [7]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [8]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

clean 78.7  linf 48.4  l2 60.1  l1 22.4  union 22.3   (123 s)


{'clean': 78.7, 'linf': 48.4, 'l2': 60.1, 'l1': 22.4, 'union': 22.3}

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [9]:
def synchronize_device():
    if device.type == "cuda":
        torch.cuda.synchronize()


def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128,
              gp_beta=None, seed=0, **kwargs):
    # Same updates and defaults as the starter; instrumentation records epoch time.
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, epochs * (len(x_train) // batch_size)
    )
    params = list(model.parameters())
    history = []
    for epoch in range(epochs):
        synchronize_device()
        start = time.perf_counter()
        losses = []
        state = getattr(loss_fn, "state", None)
        start_counts = list(state.counts) if state is not None else None
        first_record = len(state.records) if state is not None else 0
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        synchronize_device()
        row = {
            "epoch": epoch + 1, "loss": float(np.mean(losses)),
            "seconds": float(time.perf_counter() - start),
            "lr": float(opt.param_groups[0]["lr"]),
        }
        if state is not None:
            probabilities = state.probabilities()
            for j, norm in enumerate(NORMS):
                row[f"sampled_{norm}_batches"] = state.counts[j] - start_counts[j]
                row[f"estimated_failure_{norm}"] = float(state.failure_ema[j].item())
                row[f"p_{norm}"] = float(probabilities[j].item())
            epoch_records = state.records[first_record:]
            row["ce"] = float(np.mean([r["ce"] for r in epoch_records]))
            row["guard"] = float(np.mean([r["guard"] for r in epoch_records]))
        history.append(row)
        print(f"epoch {epoch + 1}: loss {row['loss']:.3f}, time {row['seconds']:.1f} s")
    model.training_history = history
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [10]:
def single_norm_loss(model, x, y, norm):
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [11]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

epoch 1: loss 1.203, time 43.9 s
epoch 2: loss 1.181, time 43.3 s
epoch 3: loss 1.151, time 43.6 s
epoch 4: loss 1.126, time 43.2 s
epoch 5: loss 1.097, time 43.1 s
epoch 6: loss 1.068, time 43.4 s
epoch 7: loss 1.048, time 43.5 s
epoch 8: loss 1.030, time 43.6 s
epoch 9: loss 1.013, time 43.6 s
epoch 10: loss 1.007, time 43.6 s
clean 78.8  linf 45.0  l2 57.0  l1 20.4  union 20.3   (133 s)


{'clean': 78.8, 'linf': 45.0, 'l2': 57.0, 'l1': 20.4, 'union': 20.3}

In [12]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

epoch 1: loss 0.892, time 43.5 s
epoch 2: loss 0.820, time 43.6 s
epoch 3: loss 0.781, time 43.6 s
epoch 4: loss 0.748, time 43.7 s
epoch 5: loss 0.708, time 43.7 s
epoch 6: loss 0.669, time 43.7 s
epoch 7: loss 0.625, time 43.6 s
epoch 8: loss 0.604, time 43.7 s
epoch 9: loss 0.584, time 43.7 s
epoch 10: loss 0.577, time 43.7 s
clean 83.0  linf 31.9  l2 60.7  l1 27.5  union 24.4   (133 s)


{'clean': 83.0, 'linf': 31.9, 'l2': 60.7, 'l1': 27.5, 'union': 24.4}

In [13]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

epoch 1: loss 1.233, time 43.5 s
epoch 2: loss 0.831, time 43.6 s
epoch 3: loss 0.596, time 43.6 s
epoch 4: loss 0.430, time 43.6 s
epoch 5: loss 0.346, time 43.6 s
epoch 6: loss 0.271, time 43.6 s
epoch 7: loss 0.223, time 43.6 s
epoch 8: loss 0.185, time 43.7 s
epoch 9: loss 0.166, time 43.7 s
epoch 10: loss 0.162, time 43.7 s
clean 85.1  linf 0.0  l2 0.0  l1 23.6  union 0.0   (133 s)


{'clean': 85.1, 'linf': 0.0, 'l2': 0.0, 'l1': 23.6, 'union': 0.0}

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [14]:
def masked_logit_kl(logits_q, logits_r, correct_q):
    log_q = F.log_softmax(logits_q, dim=1)
    log_r = F.log_softmax(logits_r, dim=1)
    per_image = (log_q.exp() * (log_q - log_r)).sum(dim=1)
    mask = correct_q.to(per_image.dtype)
    return (per_image * mask).sum() / mask.sum().clamp_min(1.0)


def ramp_loss(model, x, y, lam=0.5):
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()
    logits_q = model(x_q)
    logits_r = model(x_r)
    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y, reduction="none")
    correct_q = logits_q.detach().argmax(dim=1).eq(y)
    kl = masked_logit_kl(logits_q, logits_r, correct_q)
    return torch.maximum(ce_q, ce_r).mean() + lam * kl


def gradient_projection(g_n, g_a):
    norm_n = torch.linalg.vector_norm(g_n)
    norm_a = torch.linalg.vector_norm(g_a)
    denominator = (norm_n * norm_a).clamp_min(torch.finfo(g_n.dtype).tiny)
    cosine = ((g_n * g_a).sum() / denominator).clamp(-1.0, 1.0)
    return cosine.clamp_min(0.0) * g_n


In [15]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)

epoch 1: loss 1.432, time 92.9 s
epoch 2: loss 1.331, time 93.0 s
epoch 3: loss 1.304, time 93.3 s
epoch 4: loss 1.281, time 93.2 s
epoch 5: loss 1.254, time 93.0 s
epoch 6: loss 1.227, time 93.2 s
epoch 7: loss 1.199, time 93.3 s
epoch 8: loss 1.186, time 92.9 s
epoch 9: loss 1.171, time 92.4 s
epoch 10: loss 1.166, time 92.3 s
clean 79.2  linf 40.9  l2 61.9  l1 44.7  union 39.1   (132 s)


{'clean': 79.2, 'linf': 40.9, 'l2': 61.9, 'l1': 44.7, 'union': 39.1}

# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [16]:
def true_class_margin(logits, labels):
    true = logits.gather(1, labels[:, None]).squeeze(1)
    others = logits.masked_fill(
        F.one_hot(labels, num_classes=logits.shape[1]).bool(), float("-inf")
    ).amax(dim=1)
    return true - others


class AdaptiveNormGuard:
    def __init__(self, teacher, use_guard=True, adaptive=True,
                 decay=0.9, temperature=0.25, exploration=0.2, guard_weight=0.5):
        self.teacher = teacher.eval()
        for parameter in self.teacher.parameters():
            parameter.requires_grad_(False)
        self.use_guard = use_guard
        self.adaptive = adaptive
        self.decay = decay
        self.temperature = temperature
        self.exploration = exploration
        self.guard_weight = guard_weight
        teacher_device = next(self.teacher.parameters()).device
        self.failure_ema = torch.full((3,), 0.5, device=teacher_device)
        self.counts = [0, 0, 0]
        self.records = []

    def probabilities(self):
        if not self.adaptive:
            return torch.full_like(self.failure_ema, 1.0 / 3)
        priority = torch.softmax(self.failure_ema / self.temperature, dim=0)
        return (1 - self.exploration) * priority + self.exploration / 3

    def __call__(self, model, x, y):
        probabilities = self.probabilities()
        norm_index = int(torch.multinomial(probabilities, 1).item())
        norm = NORMS[norm_index]
        model.eval()
        x_adv = attack(model, x, y, norm)
        model.train()
        logits = model(x_adv)
        ce = F.cross_entropy(logits, y)
        guard = logits.sum() * 0.0

        if self.use_guard:
            with torch.no_grad():
                base_margin = true_class_margin(self.teacher(x_adv), y)
            margin = true_class_margin(logits, y)
            target = base_margin.clamp_min(0.0)
            shortfall = F.relu(target - margin) / (1.0 + target)
            guard = (base_margin.gt(0).to(logits.dtype) * shortfall.square()).mean()

        with torch.no_grad():
            failure = logits.detach().argmax(1).ne(y).float().mean()
            self.failure_ema[norm_index] = (
                self.decay * self.failure_ema[norm_index]
                + (1 - self.decay) * failure
            )
        self.counts[norm_index] += 1
        self.records.append({
            "batch": len(self.records) + 1, "norm": norm,
            "ce": float(ce.detach().item()),
            "guard": float(guard.detach().item()),
            "failure": float(failure.item()),
            **{f"p_{n}": float(probabilities[j].item()) for j, n in enumerate(NORMS)},
        })
        return ce + self.guard_weight * guard


def configure_my_loss(use_guard=True, adaptive=True, teacher=None):
    if teacher is None:
        teacher = load_base_model()
    my_loss.state = AdaptiveNormGuard(teacher, use_guard=use_guard, adaptive=adaptive)
    return my_loss.state


def my_loss(model, x, y):
    if getattr(my_loss, "state", None) is None:
        configure_my_loss()
    return my_loss.state(model, x, y)


In [17]:
model_mine = adv_train(load_base_model(), my_loss)
test_model(model_mine)

epoch 1: loss 1.245, time 45.6 s
epoch 2: loss 1.218, time 45.6 s
epoch 3: loss 1.176, time 45.7 s
epoch 4: loss 1.129, time 45.8 s
epoch 5: loss 1.094, time 45.7 s
epoch 6: loss 1.020, time 45.5 s
epoch 7: loss 1.027, time 45.4 s
epoch 8: loss 0.984, time 45.3 s
epoch 9: loss 0.982, time 45.3 s
epoch 10: loss 0.949, time 45.3 s
clean 78.0  linf 41.6  l2 62.8  l1 47.5  union 40.0   (132 s)


{'clean': 78.0, 'linf': 41.6, 'l2': 62.8, 'l1': 47.5, 'union': 40.0}

In [ ]:
# PART (c) — Two-seed experiments and ablation.
# Uses the model, attacks, adv_train, ramp_loss and configure_my_loss defined above.

import math
from pathlib import Path
import pandas as pd
from IPython.display import display

C_DIR = Path("hw3_part_c_outputs")
C_DIR.mkdir(exist_ok=True)

FORCE_RERUN_C = False
PART_C_VERSION = 1  # Increase this if you change your method.

C_LABELS = {
    "ramp": "RAMP",
    "mine": "My method",
    "no_guard": "Without margin guard",
    "ramp_matched": "RAMP: matched training time",
}


def run_part_c(method, seed, epochs=10):
    path = C_DIR / f"{method}_seed{seed}_epochs{epochs}.pt"

    config = {
        "version": PART_C_VERSION,
        "method": method,
        "seed": seed,
        "epochs": epochs,
        "eps": dict(EPS),
        "torch": str(torch.__version__),
        "device": str(device),
        "gpu": (
            torch.cuda.get_device_name(device)
            if device.type == "cuda" else None
        ),
    }

    if path.exists() and not FORCE_RERUN_C:
        saved = torch.load(path, map_location="cpu", weights_only=True)
        if saved.get("config") == config:
            print(f"Loaded: {C_LABELS[method]}, seed={seed}")
            return saved

    print(f"\nTraining {C_LABELS[method]}, seed={seed}, epochs={epochs}")
    trained_model = load_base_model()

    if method in ("ramp", "ramp_matched"):
        trained_model = adv_train(
            trained_model,
            ramp_loss,
            epochs=epochs,
            gp_beta=0.5,
            seed=seed,
        )
    else:
        configure_my_loss(
            use_guard=(method == "mine"),
            adaptive=True,
        )
        trained_model = adv_train(
            trained_model,
            my_loss,
            epochs=epochs,
            seed=seed,
        )

    history = trained_model.training_history
    accuracy = test_model(trained_model, steps=50, restarts=2)

    saved = {
        "config": config,
        "method": method,
        "seed": seed,
        "epochs": epochs,
        "accuracy": accuracy,
        "history": history,
        "training_seconds": sum(float(h["seconds"]) for h in history),
        "state_dict": {
            key: value.detach().cpu()
            for key, value in trained_model.state_dict().items()
        },
    }

    torch.save(saved, path)
    pd.DataFrame(history).to_csv(
        C_DIR / f"{method}_seed{seed}_epochs{epochs}_history.csv",
        index=False,
    )
    return saved


print("Evaluating the supplied base model")
base_accuracy_c = test_model(load_base_model(), steps=50, restarts=2)

runs_c = []
for seed in (0, 1):
    for method in ("ramp", "mine", "no_guard"):
        runs_c.append(run_part_c(method, seed))

Evaluating the supplied base model
clean 78.7  linf 48.4  l2 60.1  l1 22.4  union 22.3   (133 s)

Training RAMP, seed=0, epochs=10
epoch 1: loss 1.432, time 92.9 s
epoch 2: loss 1.330, time 92.8 s
epoch 3: loss 1.303, time 92.7 s


In [ ]:
# PART (c) — Training-budget comparison, results and epoch times.

def mean_training_time_c(method):
    selected = [r for r in runs_c if r["method"] == method]
    return sum(r["training_seconds"] for r in selected) / len(selected)


mine_seconds_c = mean_training_time_c("mine")
ramp_seconds_c = mean_training_time_c("ramp")

print(f"My method: {mine_seconds_c:.1f} seconds mean training time")
print(f"RAMP:      {ramp_seconds_c:.1f} seconds mean training time")

if mine_seconds_c > ramp_seconds_c:
    matched_epochs_c = max(
        11,
        math.ceil(10 * mine_seconds_c / max(ramp_seconds_c, 1e-9)),
    )

    while True:
        matched_runs_c = [
            run_part_c("ramp_matched", seed, epochs=matched_epochs_c)
            for seed in (0, 1)
        ]

        matched_seconds_c = (
            sum(r["training_seconds"] for r in matched_runs_c) / 2
        )

        if matched_seconds_c >= mine_seconds_c:
            runs_c.extend(matched_runs_c)
            break

        matched_epochs_c = max(
            matched_epochs_c + 1,
            math.ceil(
                matched_epochs_c * mine_seconds_c
                / max(matched_seconds_c, 1e-9)
            ),
        )
else:
    print("My method is faster; no extra time-matched RAMP run is needed.")

rows_c = []
epoch_rows_c = []

for run in runs_c:
    rows_c.append({
        "Method": C_LABELS[run["method"]],
        "Seed": run["seed"],
        "Epochs": run["epochs"],
        **run["accuracy"],
        "Total training seconds": run["training_seconds"],
        "Seconds per epoch": run["training_seconds"] / run["epochs"],
    })

    for h in run["history"]:
        epoch_rows_c.append({
            "Method": C_LABELS[run["method"]],
            "Seed": run["seed"],
            "Epoch": h["epoch"],
            "Seconds": h["seconds"],
        })

per_seed_c = pd.DataFrame(rows_c)
per_seed_c.to_csv(C_DIR / "part_c_per_seed_results.csv", index=False)

print("\nResults for each seed")
display(per_seed_c.round(2))

metrics_c = [
    "clean", "linf", "l2", "l1", "union",
    "Total training seconds", "Seconds per epoch",
]

means_c = (
    per_seed_c.groupby("Method", sort=False)[metrics_c]
    .mean()
    .reset_index()
)

union_sd_c = per_seed_c.groupby("Method")["union"].std()
means_c["Union sample SD"] = means_c["Method"].map(union_sd_c)

base_row_c = {
    "Method": "Base",
    **base_accuracy_c,
    "Total training seconds": float("nan"),
    "Seconds per epoch": float("nan"),
    "Union sample SD": float("nan"),
}

comparison_c = pd.concat(
    [pd.DataFrame([base_row_c]), means_c],
    ignore_index=True,
)

print("\nMean results over seeds 0 and 1")
display(comparison_c.round(2))
comparison_c.to_csv(C_DIR / "part_c_mean_results.csv", index=False)

epoch_times_c = pd.DataFrame(epoch_rows_c)
epoch_times_c.to_csv(C_DIR / "part_c_all_epoch_times.csv", index=False)

mean_epoch_times_c = (
    epoch_times_c.groupby(["Method", "Epoch"], sort=False)["Seconds"]
    .mean()
    .unstack("Epoch")
)

print("\nMean time for every epoch over seeds 0 and 1")
display(mean_epoch_times_c.round(2))
mean_epoch_times_c.to_csv(C_DIR / "part_c_mean_epoch_times.csv")

indexed_means_c = means_c.set_index("Method")

print(
    "\nUnion improvement over RAMP:",
    round(
        indexed_means_c.loc["My method", "union"]
        - indexed_means_c.loc["RAMP", "union"],
        2,
    ),
    "percentage points",
)

print(
    "Union improvement from adding the margin guard:",
    round(
        indexed_means_c.loc["My method", "union"]
        - indexed_means_c.loc["Without margin guard", "union"],
        2,
    ),
    "percentage points",
)

In [ ]:
# PART (c) —Stronger attacks on the best proposed-method model.

best_run_c = max(
    [r for r in runs_c if r["method"] == "mine"],
    key=lambda r: (
        r["accuracy"]["union"],
        r["accuracy"]["clean"],
        -r["seed"],
    ),
)

print("Best proposed-method seed:", best_run_c["seed"])

strong_path_c = C_DIR / f"mine_seed{best_run_c['seed']}_strong_evaluation.pt"
strong_config_c = {
    "training": best_run_c["config"],
    "steps": 200,
    "restarts": 5,
}

strong_saved_c = None

if strong_path_c.exists() and not FORCE_RERUN_C:
    cached_c = torch.load(
        strong_path_c, map_location="cpu", weights_only=True
    )
    if cached_c.get("config") == strong_config_c:
        strong_saved_c = cached_c

if strong_saved_c is None:
    best_model_c = make_model()
    best_model_c.load_state_dict(best_run_c["state_dict"])

    strong_accuracy_c = test_model(
        best_model_c,
        steps=200,
        restarts=5,
    )

    strong_saved_c = {
        "config": strong_config_c,
        "accuracy": strong_accuracy_c,
    }
    torch.save(strong_saved_c, strong_path_c)

strong_comparison_c = pd.DataFrame([
    {
        "Evaluation": "50 steps, 2 restarts",
        **best_run_c["accuracy"],
    },
    {
        "Evaluation": "200 steps, 5 restarts",
        **strong_saved_c["accuracy"],
    },
])

display(strong_comparison_c.round(2))
strong_comparison_c.to_csv(
    C_DIR / "part_c_stronger_attack_comparison.csv",
    index=False,
)

print(
    "Union accuracy drop under stronger attacks:",
    round(
        best_run_c["accuracy"]["union"]
        - strong_saved_c["accuracy"]["union"],
        2,
    ),
    "percentage points",
)

print("Saved part (c) results in:", C_DIR.resolve())